# Clickbait Spoiling Improvement Pipeline (Version 3) — Step 1: Environment Setup

This cell clones the repository directly from the active `phase3-qa-improvement` branch.


In [ ]:
# Clone the repository and switch to the Version 3 active development branch
!git clone https://github.com/aryonmt/clickbait-spoiling-IR-final-project.git
%cd clickbait-spoiling-IR-final-project
!git checkout phase3-qa-improvement

# Step 2: Install Project Requirements

Installs all data, modeling, and evaluation packages to ensure complete reproducibility.


In [ ]:
!pip install -r requirements.txt

# Step 3: Kaggle Inputs Absolute Path Alignment

Locks the working directory to the project folder and links Kaggle datasets to the standard data directory.


In [ ]:
import os
import glob

# Force lock the active directory to avoid path drift traps
project_dir = "/kaggle/working/clickbait-spoiling-IR-final-project"
os.chdir(project_dir)
print(f"Active working directory locked to: {os.getcwd()}")

# Ensure data directory is present in the workspace root
os.makedirs("data", exist_ok=True)

# Scan Kaggle inputs for SemEval data assets
train_paths = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
val_paths = glob.glob("/kaggle/input/**/validation.jsonl", recursive=True)

if train_paths:
    dest_train = "data/train.jsonl"
    if os.path.exists(dest_train) or os.path.islink(dest_train):
        os.remove(dest_train)
    os.symlink(train_paths[0], dest_train)
    print(f"Successfully linked train dataset to: {dest_train}")
else:
    print("[WARNING] 'train.jsonl' dataset was not found in Kaggle inputs.")

if val_paths:
    dest_val = "data/validation.jsonl"
    if os.path.exists(dest_val) or os.path.islink(dest_val):
        os.remove(dest_val)
    os.symlink(val_paths[0], dest_val)
    print(f"Successfully linked validation dataset to: {dest_val}")
else:
    print("[WARNING] 'validation.jsonl' dataset was not found in Kaggle inputs.")

# Step 4: Fine-tune Task 1 Sequence Classifier (RoBERTa-base)

Trains the 3-class classification model utilizing inverse-class frequency weights for 5 epochs.


In [ ]:
!python -m main_transformer_task1 \
    --train_path data/train.jsonl \
    --val_path data/validation.jsonl \
    --model_name roberta-base \
    --output_dir results_task1_roberta \
    --epochs 5 \
    --lr 2e-5

# Step 5: Fine-tune Task 2 extractive QA utilizing answerable_token_f1

Trains the span extraction model, ranking checkpoints strictly on answerable token-level F1.


In [ ]:
!python -m main_transformer_task2_qa \
    --train_path data/train.jsonl \
    --val_path data/validation.jsonl \
    --model_name deepset/roberta-base-squad2 \
    --output_dir results_qa \
    --epochs 3 \
    --lr 3e-5

# Step 6: Execute Integrated Spoiler Prediction Pipeline

Merges both trained checkpoints and deploys the Jaccard-retrieval routing for multi-type spoilers.


In [ ]:
import os

# Lock directory state
os.chdir("/kaggle/working/clickbait-spoiling-IR-final-project")

# Since the model is explicitly saved at the root, we no longer need glob subfolder searching
task1_model = "results_task1_roberta"
task2_model = "results_qa"

print(f"Loading Task 1 classification weights from: {task1_model}")
print(f"Loading Task 2 QA weights from: {task2_model}")

cmd = f"python -m main_integrated_pipeline --val_path data/validation.jsonl --t1_checkpoint {task1_model} --t2_checkpoint {task2_model} --output_path run_integrated_pipeline.jsonl"
os.system(cmd)

# Step 7: Build Complete Phase 4 Visual Report Dashboard

Generates correct/incorrect prediction confidence distributions and packages metrics into a zip file.


In [ ]:
# Generate visual plots and HTML reports
!python -m main_generate_report \
    --predictions_path run_integrated_pipeline.jsonl \
    --val_path data/validation.jsonl

# Archive the complete reports folder into a zip file for quick download
!zip -r reports.zip reports/